# Cleaning and Preparing the Museums Dataset

Goals:
1. Identify missing values across all columns.
2. Review the columns that matter for downstream analysis: revenue, income, city, state, museum type — plus population (which is not in the source data).
3. Convert columns to types and formats that are usable for analysis, and save a cleaned dataset.

## Setup

In [3]:
import pandas as pd
import numpy as np

# low_memory=False avoids the mixed-dtype warning on ZIP / Phone / EIN columns.
df = pd.read_csv("museums.csv", low_memory=False)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

Loaded 33,072 rows × 25 columns


## 1. Missing values

Show every column with at least one missing value, sorted by how bad it is.

In [4]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
miss_table = (pd.DataFrame({"missing": missing, "missing_%": missing_pct})
              .query("missing > 0")
              .sort_values("missing", ascending=False))
miss_table

,missing,missing_%
Alternate Name,31145,94.2
Institution Name,30323,91.7
Street Address (Physical Location),23856,72.1
Zip Code (Physical Location),23851,72.1
City (Physical Location),23849,72.1
State (Physical Location),23849,72.1
Revenue,10782,32.6
Phone Number,10140,30.7
Income,10111,30.6
Tax Period,9792,29.6


**Observations:**

- `Alternate Name` (94%) and `Institution Name` (92%) are essentially empty — safe to drop.
- All four *Physical Location* columns are 72% missing. The *Administrative Location* equivalents are fully populated. We'll drop the physical-location columns and rely on administrative.
- `Revenue`, `Income`, and `Tax Period` are ~30% missing — these come from IRS Form 990 filings and not every museum files. Keep as-is, just convert sentinel zeros (next section).
- `Latitude`/`Longitude` are essentially complete (0.2% missing).

## 2. Review key columns

### State

In [5]:
state_col = "State (Administrative Location)"
print(f"Unique state codes: {df[state_col].nunique()}")
print(f"All 2-letter uppercase: {df[state_col].dropna().str.match(r'^[A-Z]{2}$').all()}")
df[state_col].value_counts().head(10)

Unique state codes: 51
All 2-letter uppercase: True


State (Administrative Location)
CA    2670
NY    2239
TX    1886
PA    1653
OH    1363
IL    1310
FL    1149
MI    1039
MA    1028
VA     958
Name: count, dtype: int64

Clean already — 2-letter uppercase codes including DC and US territories. No fix needed.

### City

In [6]:
city_col = "City (Administrative Location)"
df[city_col].dropna().sample(10, random_state=1).tolist()

['SOUTHINGTON',
 'TACOMA',
 'BATAVIA',
 'EUTAW',
 'CHATSWORTH',
 'ATLANTA',
 'FOND DU LAC',
 'AMHERST',
 'KANSAS CITY',
 'KINGFIELD']

City names are stored in **ALL CAPS**. We'll convert to Title Case for readability while preserving the original spelling of multi-word names.

### Museum Type

In [7]:
df["Museum Type"].value_counts(dropna=False)

Museum Type
HISTORIC PRESERVATION                            14861
GENERAL MUSEUM                                    8699
ART MUSEUM                                        3241
HISTORY MUSEUM                                    2284
ARBORETUM, BOTANICAL GARDEN, OR NATURE CENTER     1484
SCIENCE & TECHNOLOGY MUSEUM OR PLANETARIUM        1081
ZOO, AQUARIUM, OR WILDLIFE CONSERVATION            564
CHILDREN'S MUSEUM                                  512
NATURAL HISTORY MUSEUM                             346
Name: count, dtype: int64

Nine clean categories, no missing values. Also stored in ALL CAPS — we'll Title Case this too.

### Revenue and Income

In [8]:
for c in ["Income", "Revenue"]:
    s = df[c]
    print(f"{c}:")
    print(f"  missing : {s.isna().sum():>6,} ({s.isna().mean()*100:.1f}%)")
    print(f"  zero    : {(s == 0).sum():>6,} ({(s == 0).mean()*100:.1f}%)")
    print(f"  negative: {(s < 0).sum():>6,}")
    print(f"  positive: {(s > 0).sum():>6,}")
    print(f"  median (positive): ${s[s > 0].median():,.0f}")
    print(f"  max              : ${s.max():,.0f}")
    print()

Income:
  missing : 10,111 (30.6%)
  zero    : 10,740 (32.5%)
  negative:      1
  positive: 12,220
  median (positive): $183,060
  max              : $83,181,439,574

Revenue:
  missing : 10,782 (32.6%)
  zero    : 10,783 (32.6%)
  negative:     40
  positive: 11,467
  median (positive): $156,377
  max              : $5,840,349,457



**Decision on zeros:** ~32% of rows show exactly `0` for both Income and Revenue. The IMLS dataset draws financials from IRS Form 990 filings, and many small museums either don't file or file 990-N postcards (which omit dollar amounts). A literal $0 of revenue is implausible for an operating museum, so these zeros almost certainly mean *not reported*. We'll convert them to `NaN` so they don't drag down means and medians.

**Negative values:** Revenue has 40 negative entries — these are legitimate (Form 990 allows reporting net losses on investments etc.). We'll keep them. The single negative income value is suspicious but we'll keep it and let the analyst decide.

### Population — not in this dataset

The CSV has **no population column**. For per-capita analysis we'll merge in 2020 US Census state populations as part of preparation.

## 3. Cleaning operations

Build the cleaned dataset step by step. We work on a copy `clean` so the raw `df` stays available for comparison.

### 3a. Drop near-empty and redundant columns

In [9]:
drop_cols = [
    "Alternate Name",          # 94% missing
    "Institution Name",        # 92% missing
    "Street Address (Physical Location)",  # 72% missing, redundant with admin
    "City (Physical Location)",
    "State (Physical Location)",
    "Zip Code (Physical Location)",
]
clean = df.drop(columns=drop_cols)
print(f"Dropped {len(drop_cols)} columns. Remaining: {clean.shape[1]}")

Dropped 6 columns. Remaining: 19


### 3b. Standardize column names to snake_case

In [10]:
rename_map = {
    "Museum ID": "museum_id",
    "Museum Name": "museum_name",
    "Legal Name": "legal_name",
    "Museum Type": "museum_type",
    "Street Address (Administrative Location)": "street",
    "City (Administrative Location)": "city",
    "State (Administrative Location)": "state",
    "Zip Code (Administrative Location)": "zip",
    "Phone Number": "phone",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Locale Code (NCES)": "locale_code",
    "County Code (FIPS)": "county_fips",
    "State Code (FIPS)": "state_fips",
    "Region Code (AAM)": "region_code",
    "Employer ID Number": "ein",
    "Tax Period": "tax_period",
    "Income": "income",
    "Revenue": "revenue",
}
clean = clean.rename(columns=rename_map)
list(clean.columns)

['museum_id',
 'museum_name',
 'legal_name',
 'museum_type',
 'street',
 'city',
 'state',
 'zip',
 'phone',
 'latitude',
 'longitude',
 'locale_code',
 'county_fips',
 'state_fips',
 'region_code',
 'ein',
 'tax_period',
 'income',
 'revenue']

### 3c. Fix string ID columns (ZIP, EIN, Phone)

In [11]:
# ZIP: pad to 5 digits to restore leading zeros (NJ, MA, CT etc. lose them as ints)
clean["zip"] = clean["zip"].astype("string").str.zfill(5).where(clean["zip"].notna())

# EIN: pad to 9 digits
clean["ein"] = clean["ein"].astype("string").str.zfill(9).where(clean["ein"].notna())

# Phone: keep as string, no padding (lengths vary)
clean["phone"] = clean["phone"].astype("string")

print("ZIP length distribution after fix:")
print(clean["zip"].dropna().str.len().value_counts())
print("\nEIN length distribution after fix:")
print(clean["ein"].dropna().str.len().value_counts())

ZIP length distribution after fix:
zip
5    33071
6        1
Name: count, dtype: Int64

EIN length distribution after fix:
ein
9    27554
Name: count, dtype: Int64


### 3d. Title-case city and museum type

In [12]:
clean["city"] = clean["city"].str.title()
clean["museum_type"] = clean["museum_type"].str.title()
clean[["city", "state", "museum_type"]].head()

,city,state,museum_type
0,Anchorage,AK,History Museum
1,Anchorage,AK,"Arboretum, Botanical Garden, Or Nature Center"
2,Kenai,AK,Science & Technology Museum Or Planetarium
3,Kenai,AK,Historic Preservation
4,Anchorage,AK,History Museum


### 3e. Convert Tax Period (YYYYMM float) to year and month

In [13]:
tp = clean["tax_period"].astype("Int64").astype("string")
clean["tax_year"]  = pd.to_numeric(tp.str[:4],  errors="coerce").astype("Int64")
clean["tax_month"] = pd.to_numeric(tp.str[4:6], errors="coerce").astype("Int64")
clean = clean.drop(columns=["tax_period"])
clean[["tax_year", "tax_month"]].dropna().head()

,tax_year,tax_month
0,2013,12
1,2013,12
2,2013,12
3,2014,12
4,2013,12


### 3f. Treat zero revenue/income as missing

In [14]:
for c in ["income", "revenue"]:
    n_before = clean[c].isna().sum()
    clean[c] = clean[c].where(clean[c] != 0, np.nan)
    n_after = clean[c].isna().sum()
    print(f"{c}: NaN went from {n_before:,} to {n_after:,} (+{n_after - n_before:,} converted)")

income: NaN went from 10,111 to 20,851 (+10,740 converted)
revenue: NaN went from 10,782 to 21,565 (+10,783 converted)


### 3g. Fix coordinates: drop the (0, 0) row to NaN

In [15]:
bad = (clean["latitude"] == 0) | (clean["longitude"] == 0)
print(f"Rows with lat=0 or lon=0: {bad.sum()}")
clean.loc[bad, ["latitude", "longitude"]] = np.nan

Rows with lat=0 or lon=0: 1


### 3h. Add state population column for per-capita analysis

In [16]:
# 2020 US Census state populations (incl. DC and US territories that appear in IMLS)
state_pop_2020 = {
    "AL": 5024279, "AK": 733391, "AZ": 7151502, "AR": 3011524,
    "CA": 39538223, "CO": 5773714, "CT": 3605944, "DE": 989948,
    "DC": 689545, "FL": 21538187, "GA": 10711908, "HI": 1455271,
    "ID": 1839106, "IL": 12812508, "IN": 6785528, "IA": 3190369,
    "KS": 2937880, "KY": 4505836, "LA": 4657757, "ME": 1362359,
    "MD": 6177224, "MA": 7029917, "MI": 10077331, "MN": 5706494,
    "MS": 2961279, "MO": 6154913, "MT": 1084225, "NE": 1961504,
    "NV": 3104614, "NH": 1377529, "NJ": 9288994, "NM": 2117522,
    "NY": 20201249, "NC": 10439388, "ND": 779094, "OH": 11799448,
    "OK": 3959353, "OR": 4237256, "PA": 13002700, "RI": 1097379,
    "SC": 5118425, "SD": 886667, "TN": 6910840, "TX": 29145505,
    "UT": 3271616, "VT": 643077, "VA": 8631393, "WA": 7705281,
    "WV": 1793716, "WI": 5893718, "WY": 576851,
    "PR": 3285874, "VI": 87146, "GU": 153836, "AS": 49710, "MP": 47329,
}
clean["state_population_2020"] = clean["state"].map(state_pop_2020).astype("Int64")
print(f"Rows with population matched: {clean['state_population_2020'].notna().sum():,} / {len(clean):,}")

Rows with population matched: 33,072 / 33,072


## 4. Final summary

In [17]:
print(f"Shape: {clean.shape[0]:,} rows × {clean.shape[1]} columns")
print(f"\nDtypes:")
print(clean.dtypes)

Shape: 33,072 rows × 21 columns

Dtypes:
museum_id                  int64
museum_name                  str
legal_name                   str
museum_type                  str
street                       str
city                         str
state                        str
zip                       string
phone                     string
latitude                 float64
longitude                float64
locale_code              float64
county_fips              float64
state_fips               float64
region_code                int64
ein                       string
income                   float64
revenue                  float64
tax_year                   Int64
tax_month                  Int64
state_population_2020      Int64
dtype: object


In [18]:
clean.head()

,museum_id,museum_name,legal_name,museum_type,street,city,state,zip,phone,latitude,...,locale_code,county_fips,state_fips,region_code,ein,income,revenue,tax_year,tax_month,state_population_2020
0,8400200098,ALASKA AVIATION HERITAGE MUSEUM,ALASKA AVIATION HERITAGE MUSEUM,History Museum,4721 AIRCRAFT DR,Anchorage,AK,99502,9072485325,61.17925,...,1.0,20.0,2.0,6,920071852,602912.0,550236.0,2013,12,733391
1,8400200117,ALASKA BOTANICAL GARDEN,ALASKA BOTANICAL GARDEN INC,"Arboretum, Botanical Garden, Or Nature Center",4601 CAMPBELL AIRSTRIP RD,Anchorage,AK,99507,9077703692,61.16890,...,4.0,20.0,2.0,6,920115504,1379576.0,1323742.0,2013,12,733391
2,8400200153,ALASKA CHALLENGER CENTER FOR SPACE SCIENCE TEC...,ALASKA CHALLENGER CENTER FOR SPACE SCIENCE TEC...,Science & Technology Museum Or Planetarium,9711 KENAI SPUR HWY,Kenai,AK,99611,9072832000,60.56149,...,3.0,122.0,2.0,6,921761906,740030.0,729080.0,2013,12,733391
3,8400200143,ALASKA EDUCATORS HISTORICAL SOCIETY,ALASKA EDUCATORS HISTORICAL SOCIETY,Historic Preservation,214 BIRCH STREET,Kenai,AK,99611,2142472478,60.56280,...,3.0,122.0,2.0,6,920165178,NaN,NaN,2014,12,733391
4,8400200027,ALASKA HERITAGE MUSEUM,ALASKA AVIATION HERITAGE MUSEUM,History Museum,301 W NORTHERN LIGHTS BLVD,Anchorage,AK,99503,9072652834,61.17925,...,1.0,20.0,2.0,6,920071852,602912.0,550236.0,2013,12,733391


### Save cleaned dataset

In [20]:
out_path = "museums_clean.csv"
clean.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({clean.shape[0]:,} rows × {clean.shape[1]} cols)")

Wrote museums_clean.csv (33,072 rows × 21 cols)


In [22]:
%whos

Variable         Type         Data/Info
---------------------------------------
bad              Series       Shape: (33072,)
c                str          revenue
city_col         str          City (Administrative Location)
clean            DataFrame    Shape: (33072, 21)
df               DataFrame    Shape: (33072, 25)
drop_cols        list         n=6
miss_table       DataFrame    Shape: (17, 2)
missing          Series       Shape: (25,)
missing_pct      Series       Shape: (25,)
n_after          int64        21565
n_before         int64        10782
np               module       <module 'numpy' from '/op<...>kages/numpy/__init__.py'>
out_path         str          museums_clean.csv
pd               module       <module 'pandas' from '/o<...>ages/pandas/__init__.py'>
plt              module       <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
rename_map       dict         n=19
s                Series       Shape: (33072,)
state_col        str          State (Administrative L

In [24]:
state_pop = {
    "AL": 5024279, "AK": 733391, "AZ": 7151502, "AR": 3011524,
    "CA": 39538223, "CO": 5773714, "CT": 3605944, "DE": 989948,
    "DC": 689545, "FL": 21538187, "GA": 10711908, "HI": 1455271,
    "ID": 1839106, "IL": 12812508, "IN": 6785528, "IA": 3190369,
    "KS": 2937880, "KY": 4505836, "LA": 4657757, "ME": 1362359,
    "MD": 6177224, "MA": 7029917, "MI": 10077331, "MN": 5706494,
    "MS": 2961279, "MO": 6154913, "MT": 1084225, "NE": 1961504,
    "NV": 3104614, "NH": 1377529, "NJ": 9288994, "NM": 2117522,
    "NY": 20201249, "NC": 10439388, "ND": 779094, "OH": 11799448,
    "OK": 3959353, "OR": 4237256, "PA": 13002700, "RI": 1097379,
    "SC": 5118425, "SD": 886667, "TN": 6910840, "TX": 29145505,
    "UT": 3271616, "VT": 643077, "VA": 8631393, "WA": 7705281,
    "WV": 1793716, "WI": 5893718, "WY": 576851,
}

In [25]:
counts = clean["state"].value_counts().rename("museums")
pop = pd.Series(state_pop, name="population")

stats = pd.concat([counts, pop], axis=1).dropna()
stats["per_100k"] = (stats["museums"] / stats["population"] * 100_000).round(2)
stats = stats.sort_values("per_100k", ascending=False)
stats

,museums,population,per_100k
VT,292,643077,45.41
ME,521,1362359,38.24
ND,269,779094,34.53
WY,192,576851,33.28
DC,190,689545,27.55
SD,234,886667,26.39
NH,363,1377529,26.35
MT,270,1084225,24.90
AK,162,733391,22.09
IA,661,3190369,20.72
